<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/rf_sanity_check_three_modalities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Receptive-field sanity check across three recording scales

A **confidence-building** notebook for the [OpenScope Community Predictive
Processing](https://allenneuraldynamics.github.io/openscope-community-predictive-processing/)
dataset. Before running any sophisticated mismatch/prediction-error analysis, we
verify that the whole pipeline — *stream NWB → align to stimulus trials → extract
neural response → build a receptive field (RF)* — produces sensible, retinotopically
localized RFs in **all three modalities**:

| Modality | DANDI | Signal | Spatial scale |
|---|---|---|---|
| Neuropixels ecephys | [001637](https://dandiarchive.org/dandiset/001637) | spike rate | area / layer |
| Mesoscope 2-photon | [001768](https://dandiarchive.org/dandiset/001768) | ΔF/F (jGCaMP8s, soma) | single cell |
| SLAP2 | [001424](https://dandiarchive.org/dandiset/001424) | ΔF/F (iGluSnFR, glutamate) | dendrite |

RF mapping is an ideal check because the expected answer is known: a real visual
neuron should respond to a **compact, contiguous patch of visual space**. If the
RFs come out as localized hotspots, the streaming, trial-alignment and
response-extraction code are all working.

**Runtime:** CPU is fine. Each session streams several GB over HTTP via `remfile`
(no full download). The whole notebook runs in ~5–8 min. Open in Colab and run top
to bottom.

In [ ]:
#@title Install dependencies
!pip -q install pynwb h5py remfile requests pandas numpy matplotlib

In [ ]:
#@title Streaming helper — resolve a DANDI asset to a signed S3 URL and open it
import h5py, remfile, requests, numpy as np, pandas as pd

def s3_url(dandiset, asset_id, version="draft"):
    """Follow the DANDI download redirect to the (signed, expiring) S3 URL."""
    r = requests.get(
        f"https://api.dandiarchive.org/api/dandisets/{dandiset}/versions/{version}/assets/{asset_id}/download/",
        allow_redirects=False, timeout=30)
    return r.headers["Location"]

def open_nwb(dandiset, asset_id):
    """Open a remote NWB for lazy, chunked HTTP reads (no full download)."""
    return h5py.File(remfile.File(s3_url(dandiset, asset_id)), "r")

def col(group, name):
    """Read an intervals/units column, decoding byte-strings to str."""
    v = group[name][:]
    return np.array([x.decode() if isinstance(x, bytes) else x for x in v])

print("helpers ready")

## 1 · Neuropixels ecephys — RFs from spikes

We use one standard-oddball session that also carries a `RF mapping` block. The RF
stimulus is a **9×9 grid of patch gratings** (azimuth/elevation ±40° in 10° steps),
each shown 0.25 s. For every visual unit we count spikes in a short post-onset
window (30–200 ms), subtract a pre-onset baseline, and average per screen position
to build the RF map.

Units are assigned to a CCF area/layer via the *corrected* per-probe electrode
mapping (`extremum_channel_index` is a **per-probe** index into a stacked
electrodes table — indexing it directly would assign every unit to the first probe).

In [ ]:
#@title Ecephys RF — session sub-830851 (standard oddball, CCF-labelled)
ECE_ASSET = "9b9e8abe-7b43-47f1-b8e1-4114f87898a1"   # sub-830851, 2026-03-17
fh = open_nwb("001637", ECE_ASSET)

# --- RF-mapping stimulus grid ---
g = fh["intervals"]["RF mapping_presentations"]
X = col(g, "X").astype(float); Y = col(g, "Y").astype(float)
t0 = g["start_time"][:]
xs = np.array(sorted(set(X))); ys = np.array(sorted(set(Y)))
xi = {v:k for k,v in enumerate(xs)}; yi = {v:k for k,v in enumerate(ys)}
tX = np.array([xi[v] for v in X]); tY = np.array([yi[v] for v in Y])
print(f"RF grid {len(xs)}x{len(ys)}, {len(t0)} trials")

# --- units: spikes + QC + CCF region (corrected per-probe mapping) ---
U = fh["units"]
st = U["spike_times"][:]; sti = U["spike_times_index"][:]
def spikes(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
n_units = len(sti)
qc = U["default_qc"][:] if "default_qc" in U else np.ones(n_units, bool)

E = fh["general/extracellular_ephys/electrodes"]
eloc = col(E, "location"); egroup = col(E, "group_name")
dev = col(U, "device_name"); eci = U["extremum_channel_index"][:]
groups = sorted(set(egroup), key=lambda gn: np.where(egroup==gn)[0][0])
offset = {gn: np.where(egroup==gn)[0][0] for gn in groups}
blocklen = {gn: int((egroup==gn).sum()) for gn in groups}
def grp_for_dev(d):
    for gn in groups:
        if d[-1].lower() in gn.lower(): return gn
    return groups[0]
dev2grp = {d: grp_for_dev(d) for d in set(dev)}
u_region = np.array([eloc[offset[dev2grp[dev[i]]] + min(int(eci[i]), blocklen[dev2grp[dev[i]]]-1)]
                     for i in range(n_units)])
import re
is_vis = np.array([bool(re.match(r"VIS", r)) for r in u_region])
sel = np.where(qc & is_vis)[0]
print(f"{len(sel)} visual QC units")

In [ ]:
#@title Compute ecephys RF maps and rank by quality
RESP, BASE = (0.03, 0.20), (-0.15, -0.02)   # seconds relative to patch onset

def ece_rf_map(unit_idx):
    sp = spikes(unit_idx)
    grid = np.zeros((len(ys), len(xs))); cnt = np.zeros_like(grid); base = 0.0
    for k in range(len(t0)):
        s = t0[k]
        r = np.searchsorted(sp, [s+RESP[0], s+RESP[1]]); rc = (r[1]-r[0])/(RESP[1]-RESP[0])
        b = np.searchsorted(sp, [s+BASE[0], s+BASE[1]]); bc = (b[1]-b[0])/(BASE[1]-BASE[0])
        grid[tY[k], tX[k]] += rc; cnt[tY[k], tX[k]] += 1; base += bc
    return grid/np.clip(cnt,1,None) - base/len(t0)

ece_maps = {}; rows = []
for i in sel:
    m = ece_rf_map(i); ece_maps[i] = m
    peak = m.max(); rows.append((i, peak, peak/(m.std()+1e-9)))
Qe = pd.DataFrame(rows, columns=["unit","peak_hz","snr"]).sort_values("snr", ascending=False)
Qe["region"] = [u_region[u] for u in Qe.unit]
ece_pick = Qe[(Qe.peak_hz>5)&(Qe.snr>4)].head(6)
print(ece_pick.to_string(index=False))
fh.close()

## 2 · Mesoscope 2-photon — RFs from ΔF/F (somatic)

Same 9×9 RF grid. The mesoscope images 8 planes simultaneously (VISp ×4, VISl ×4
depths). We take one plane, keep only **soma** ROIs (`is_soma` mask), and use a
slower response window (100–800 ms) because GCaMP calcium indicators are slow.

In [ ]:
#@title Mesoscope RF — session sub-832700 (standard oddball)
MESO_ASSET = "83e0c8f3-5208-417c-87c4-bc4617b0f834"   # sub-832700, 2026-01-31
fhm = open_nwb("001768", MESO_ASSET)
gm = fhm["intervals"]["RF mapping_presentations"]
Xm = col(gm,"X").astype(float); Ym = col(gm,"Y").astype(float); t0m = gm["start_time"][:]
xsm = np.array(sorted(set(Xm))); ysm = np.array(sorted(set(Ym)))
tXm = np.array([{v:k for k,v in enumerate(xsm)}[v] for v in Xm])
tYm = np.array([{v:k for k,v in enumerate(ysm)}[v] for v in Ym])

PLANE = "VISl_4"   # any of VISp_0..3, VISl_4..7
p0 = fhm["processing"][PLANE]
dff = p0["dff_timeseries"]["dff_timeseries"]
data = dff["data"][:]; tsv = dff["timestamps"][:]
is_soma = p0["image_segmentation"]["roi_table"]["is_soma"][:].astype(bool)
soma_idx = np.where(is_soma)[0]
print(f"plane {PLANE}: {data.shape[1]} ROIs, {len(soma_idx)} soma, {data.shape[0]} frames")

RESP_M, BASE_M = (0.1, 0.8), (-0.3, 0.0)
def meso_rf_map(roi):
    tr = data[:, roi]; grid = np.zeros((len(ysm), len(xsm))); cnt = np.zeros_like(grid)
    for k in range(len(t0m)):
        s = t0m[k]
        rm = (tsv>=s+RESP_M[0])&(tsv<s+RESP_M[1]); bm = (tsv>=s+BASE_M[0])&(tsv<s+BASE_M[1])
        if rm.sum()<1 or bm.sum()<1: continue
        grid[tYm[k], tXm[k]] += np.nanmean(tr[rm]) - np.nanmean(tr[bm]); cnt[tYm[k], tXm[k]] += 1
    return grid/np.clip(cnt,1,None)

meso_maps = {}; rows = []
for r in soma_idx:
    m = meso_rf_map(r); meso_maps[r] = m
    peak = np.nanmax(m); rows.append((r, peak, peak/(np.nanstd(m)+1e-9)))
Qm = pd.DataFrame(rows, columns=["roi","peak_dff","snr"]).sort_values("snr", ascending=False)
meso_pick = Qm[(Qm.peak_dff>0.3)&(Qm.snr>4.5)].head(6)
print(meso_pick.to_string(index=False))
fhm.close()

## 3 · SLAP2 — RFs from glutamate (iGluSnFR), dendritic resolution

SLAP2 stores all stimuli in one monolithic `gratings` table. RF trials are the
small-diameter (20°) patches on a **7×7 grid** (±30°). Two important SLAP2 quirks,
both learned the hard way:

1. **Timing offset** — each DMD path has a fixed onset delay (DMD1 **+0.115 s**).
2. **Corrupt timestamps** — DMD1's stored timestamps are compressed (they run to
   ~1000 s when the true recording is ~3020 s). Because the two DMDs image
   *simultaneously*, we rebuild DMD1's timebase as a uniform axis over DMD2's
   (intact) acquisition span. Without this fix, the RF trials fall outside the
   apparent timestamp range and every response comes back empty.

In [ ]:
#@title SLAP2 RF — session sub-801381 / DMD1 (glutamate)
SLAP_ASSET = "d5120d10"   # sub-801381, 2025-06-05
fhs = open_nwb("001424", SLAP_ASSET)
gs = fhs["intervals/gratings"]
xg = gs["x"][:]; yg = gs["y"][:]; dia = gs["diameter"][:]; t0s = gs["start_time"][:]
is_rf = dia < 30                       # small patches = RF mapping
rf_idx = np.where(is_rf)[0]
xg_r, yg_r = xg[rf_idx], yg[rf_idx]
xsg = np.array(sorted(set(xg_r))); ysg = np.array(sorted(set(yg_r)))
tXg = np.array([{v:k for k,v in enumerate(xsg)}[v] for v in xg_r])
tYg = np.array([{v:k for k,v in enumerate(ysg)}[v] for v in yg_r])

# DMD1 dFF + timebase rebuild
dff1 = fhs["processing/ophys/Fluorescence_DMD1/DMD1_dFF"]
d = dff1["data"][:]
dff2 = fhs["processing/ophys/Fluorescence_DMD2/DMD2_dFF"]      # intact timebase reference
tsv2 = dff2["timestamps"][:]
tsv1_fix = np.linspace(tsv2[0], tsv2[-1], d.shape[0])          # simultaneous acquisition
print(f"DMD1 {d.shape[1]} ROIs; timebase rebuilt to {tsv1_fix[-1]:.0f}s (stored ts topped out at {dff1['timestamps'][-1]:.0f}s)")

DMD1_OFFSET = 0.115
t0_r = t0s[rf_idx] + DMD1_OFFSET
RESP_S, BASE_S = (0.05, 0.35), (-0.25, -0.02)   # ISI 700ms -> window <=0.35s; glutamate is fast
def slap_rf_map(roi):
    tr = d[:, roi]; grid = np.zeros((len(ysg), len(xsg))); cnt = np.zeros_like(grid)
    for k in range(len(t0_r)):
        s = t0_r[k]
        rm = (tsv1_fix>=s+RESP_S[0])&(tsv1_fix<s+RESP_S[1]); bm = (tsv1_fix>=s+BASE_S[0])&(tsv1_fix<s+BASE_S[1])
        if rm.sum()<1 or bm.sum()<1: continue
        rv, bv = tr[rm], tr[bm]
        if not (np.isfinite(rv).any() and np.isfinite(bv).any()): continue
        val = np.nanmean(rv) - np.nanmean(bv)
        if np.isfinite(val):
            grid[tYg[k], tXg[k]] += val; cnt[tYg[k], tXg[k]] += 1
    return grid/np.clip(cnt,1,None)

slap_maps = {}; rows = []
for r in range(d.shape[1]):
    m = slap_rf_map(r); slap_maps[r] = m
    peak = np.nanmax(m); rows.append((r, peak, peak/(np.nanstd(m)+1e-9)))
Qs = pd.DataFrame(rows, columns=["roi","peak_dff","snr"]).sort_values("snr", ascending=False)
slap_pick = Qs[(Qs.peak_dff>0.15)&(Qs.snr>2.8)].head(6)
print(slap_pick.to_string(index=False))
fhs.close()

## 4 · The figure — example RFs across all three scales

Each row is a modality; each panel is one unit/ROI, showing the baseline-subtracted
response averaged per screen position. Rows use **independent colour scales**
(spikes in Hz vs ΔF/F are not directly comparable in magnitude) — this is a
"do the RFs look right" check, not a cross-modal magnitude comparison.

In [ ]:
#@title Plot
import matplotlib.pyplot as plt
rows_spec = [("Neuropixels\nspikes",       xs,  ys,  ece_maps,  ece_pick,  "unit", "region",  "peak_hz",  "Hz",   "#08519c"),
             ("Mesoscope 2p\nΔF/F (soma)",  xsm, ysm, meso_maps, meso_pick, "roi",  None,      "peak_dff", "ΔF/F", "#238b45"),
             ("SLAP2\nglutamate ΔF/F",      xsg, ysg, slap_maps, slap_pick, "roi",  None,      "peak_dff", "ΔF/F", "#d94801")]
ncol = 6
fig, axes = plt.subplots(3, ncol, figsize=(13, 7.4))
for ri,(rt,xg_,yg_,maps,pick,idc,regc,peakc,unit,color) in enumerate(rows_spec):
    ids = list(pick[idc])[:ncol]
    for ci in range(ncol):
        ax = axes[ri, ci]
        if ci < len(ids):
            row = pick[pick[idc]==ids[ci]].iloc[0]; m = maps[ids[ci]]
            ax.imshow(m, origin="lower", extent=[xg_[0],xg_[-1],yg_[0],yg_[-1]],
                      cmap="magma", vmin=0, vmax=np.nanmax(m), aspect="equal")
            ax.set_xticks([xg_[0],0,xg_[-1]]); ax.set_yticks([yg_[0],0,yg_[-1]]); ax.tick_params(labelsize=6)
            lab = row[regc] if regc else PLANE if ri==1 else "DMD1"
            ax.set_title(f"{lab} · {row[peakc]:.2f} {unit}", fontsize=6.8, pad=3)
            ax.axhline(0,color="w",lw=0.4,alpha=0.4); ax.axvline(0,color="w",lw=0.4,alpha=0.4)
        else:
            ax.axis("off")
    axes[ri,0].set_ylabel("elevation (°)", fontsize=7.5)
for ci in range(ncol): axes[2,ci].set_xlabel("azimuth (°)", fontsize=7.5)
for ri,(rt,*rest) in enumerate(rows_spec):
    bb = axes[ri,0].get_position()
    fig.text(0.015, (bb.y0+bb.y1)/2, rt, rotation=90, va="center", ha="center",
             fontsize=9.5, fontweight="bold", color=rest[-1])
fig.suptitle("Example receptive fields across three recording scales", fontsize=12, y=0.975)
fig.text(0.5, 0.935, "OpenScope Predictive Processing · RF-mapping block (patch gratings on a visual-space grid)",
         ha="center", fontsize=8.5, color="#444")
fig.subplots_adjust(left=0.075, right=0.985, top=0.895, bottom=0.09, wspace=0.35, hspace=0.5)
fig.savefig("rf_examples_three_modalities.png", dpi=200, bbox_inches="tight")
plt.show()

## Takeaway

All three modalities yield compact, retinotopically localized receptive fields:

- **Ecephys** — textbook single-hotspot RFs, peak evoked rates 6–30 Hz.
- **Mesoscope** — clean single-lobe somatic RFs (most compact, as expected for cell bodies).
- **SLAP2** — localized but noisier glutamate RFs, consistent with lower-SNR dendritic signals.

The pipeline (streaming → trial alignment → response extraction) is validated in all
three, so we can trust it for the more sophisticated prediction-error / mismatch
analyses.